In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load & Preprocess the Data

In [ ]:
%%capture
!pip install transformers datasets accelerate -q
!pip install evaluate

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
en_train  = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/en_train.csv')
en_dev = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/en_dev.csv')
en_test = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/TestWithNoLabel/TestWithNoLabel/en_test_without_labels.csv')
en_train

In [ ]:
label_mapping = {'Hope': 1, 'Not Hope': 0}

# Apply label encoding
en_train['binary'] = en_train['binary'].map(label_mapping)
en_dev['binary'] = en_dev['binary'].map(label_mapping)

# Drop the 'multiclass' column
en_train.drop(columns=['multiclass'], inplace=True)
en_dev.drop(columns=['multiclass'], inplace=True)

# Drop NaN values if any
en_train.dropna(inplace=True)
en_dev.dropna(inplace=True)

In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Tokenize training data and get lengths
train_text_lengths = [len(tokenizer.tokenize(text)) for text in en_train['text']]

# Plot the distribution
plt.figure(figsize=(8,5))
plt.hist(train_text_lengths, bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Tokens")
plt.ylabel("Frequency")
plt.title("Token Length Distribution in Training Data")
plt.show()


In [ ]:
# Use the percentile method:
Max_length = int(np.percentile(train_text_lengths, 97))  # Covers 96% of texts
Max_length

# Tokenization with XLM-RoBERTa

In [ ]:
# Tokenization function
def tokenize_data(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=Max_length)

# Apply tokenization
train_texts = en_train["text"].tolist()
train_labels = en_train["binary"].tolist()

dev_texts = en_dev["text"].tolist()
dev_labels = en_dev["binary"].tolist()

train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=256)
dev_encodings = tokenizer(dev_texts, padding=True, truncation=True, max_length=256)

# Convert Data to PyTorch Dataset

In [ ]:
import torch

class HopeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = HopeDataset(train_encodings, train_labels)
dev_dataset = HopeDataset(dev_encodings, dev_labels)


# Load Pre-Trained Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2)


#  Define Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)


# Define Metrics for Evaluation

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1_score["f1"]}


# Initialize Trainer and Fine-Tune the Model

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluate the Model

In [ ]:
trainer.evaluate()


# Save the Model

In [ ]:
output_dir = "/kaggle/working/xlm-roberta-hope"

# Save the fine-tuned model
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)


In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the fine-tuned model and tokenizer
model_path = "/kaggle/working/xlm-roberta-hope"  # Update this path if necessary
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()  # Set model to evaluation mode

# Force running on CPU
device = torch.device("cpu")
model.to(device)

# Ensure 'en_test' DataFrame exists
if "en_test" not in globals():
    raise ValueError("❌ 'en_test' DataFrame is not defined.")

test_texts = en_test["text"].tolist()  # Ensure the test CSV has a 'text' column

# Tokenize test data (max_length = 85 based on analysis)
max_length = Max_length  # Update this if defined elsewhere
encoded_inputs = tokenizer(
    test_texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt"
)

# Move data to CPU
encoded_inputs = {key: val.to(device) for key, val in encoded_inputs.items()}

# Make predictions
with torch.no_grad():
    outputs = model(**encoded_inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).numpy()

# Convert predictions to original labels
label_map = {0: "Not Hope", 1: "Hope"}  # Update if dataset labels differ
predicted_labels = [label_map[pred] for pred in predictions]

# Save predictions to CSV
submission_df = pd.DataFrame({"Text": test_texts, "Tag": predicted_labels})
submission_df.to_csv("predictions.csv", index=False)

print("✅ Predictions successfully saved to 'predictions.csv' using CPU.")


In [ ]:
pd.read_csv('/kaggle/working/predictions.csv')